In [1]:
# fetch the data

import httpx
import pandas as pd

URL = "https://openapi.twse.com.tw/v1/opendata/t187ap04_L"

resp = httpx.get(URL, timeout=30, headers={"Accept": "application/json"})
print(resp.status_code, resp.headers.get("content-type"), resp.elapsed)

records = resp.json()
print(type(records), len(records))

200 application/json 0:00:01.591636
<class 'list'> 401


In [2]:
# check the fields and first 3 records

print(list(records[0].keys()))   # the raw names show any stray spaces in them

df = pd.DataFrame(records)
df.columns = df.columns.str.strip()
df.head(3).T

['出表日期', '發言日期', '發言時間', '公司代號', '公司名稱', '主旨 ', '符合條款', '事實發生日', '說明']


,0,1,2
出表日期,1151107,1151107,1151107
發言日期,1151106,1151106,1151106
發言時間,1820,70004,101814
公司代號,3045,2072,1904
公司名稱,台灣大,世紀能源,正隆
主旨,代重要子公司台信電訊(股)公司公告公開收購精誠資訊\r\n(股)公司普通股之公開收購期間屆滿,公告本公司名稱由「世紀離岸風電設備股份有限公司」更名\r\n為「世紀能源設備股份有限公司」，...,公告本公司大園廠C廠區戶外料廠發生火災
符合條款,第38款,第51款,第51款
事實發生日,1151105,1150824,1151106
說明,1.公開收購期間屆滿之日期:115/11/05\n2.公開收購人之公司名稱:\n台信電訊股份...,1.事實發生日：民國115年08月24日\r\n2.公司名稱：世紀能源設備股份有限公司(原名...,1.事實發生日:115/11/06\n2.公司名稱:正隆股份有限公司\n3.與公司關係(請輸...


In [3]:
# check for blanks and duplicates

blank = df.astype(str).apply(lambda s: s.str.strip().eq("") | s.eq("None"))
print("Blank values per column:\n", blank.sum())
print("Fully duplicated rows:", df.duplicated().sum())

key = ["公司代號", "發言日期", "發言時間", "主旨"]   # adjust to the names from Cell 2
print("Duplicates on candidate key:", df.duplicated(subset=key).sum())

Blank values per column:
 出表日期     0
發言日期     0
發言時間     0
公司代號     0
公司名稱     0
主旨       0
符合條款     0
事實發生日    0
說明       0
dtype: int64
Fully duplicated rows: 0
Duplicates on candidate key: 0


In [4]:
# check dates, text length and categories

date_cols = ["出表日期", "發言日期", "事實發生日"]   # adjust to the names from Cell 2
print(df[date_cols].head(5))
for col in date_cols:
    print(col, "distinct values:", df[col].nunique())

print(df["說明"].str.len().describe())
print(df["符合條款"].value_counts().head(15))

      出表日期     發言日期    事實發生日
0  1151107  1151106  1151105
1  1151107  1151106  1150824
2  1151107  1151106  1151106
3  1151107  1151106  1151106
4  1151107  1151106  1151113
出表日期 distinct values: 1
發言日期 distinct values: 1
事實發生日 distinct values: 14
count     401.000000
mean      589.670823
std       582.990769
min        56.000000
25%       313.000000
50%       446.000000
75%       575.000000
max      6484.000000
Name: 說明, dtype: float64
符合條款
第31款    132
第51款     44
第20款     42
第8款      30
第11款     29
第12款     22
第6款      20
第23款     20
第22款     13
第14款      7
第21款      7
第43款      7
第26款      6
第15款      6
第17款      4
Name: count, dtype: int64


# Observations

**Question:** What are the exact field names and how many records came back?

**Answer:** The exact field names are 出表日期, 發言日期, 發言時間, 公司代號, 公司名稱, 主旨 , 符合條款, 事實發生日, and 說明. 401 records came back. The raw name of 主旨 has a trailing space, so strip column names on ingest.

**Question:** Are dates in ROC format (e.g. 1151106) or Western format?

**Answer:** Dates are in ROC format.

**Question:** Does the response cover one day or several?

**Answer:** Only the latest day.

**Question:** Which fields can uniquely identify an announcement?

**Answer:** \[公司代號, 發言日期, 發言時間, 主旨 \]. Candidate key: no duplicates in this one-day snapshot; recheck across several days.

**Question:** How long is 說明 at the median and maximum?

**Answer:** Median length is 446 in characters. Maximum length is 6484 in characters.

**Question:** What categories does 符合條款 show?

**Answer:** 第31款, 第51款, 第20款, 第8款, 第11款, 第12款, 第6款, 第23款, 第22款, 第14款, 第21款, 第43款, 第26款, 第15款, and 第17款. Top 15 shown (389 of 401 records); 12 records fall in other clauses.

These all affect the next steps:

* **Time format:** 發言時間 drops leading zeros: 1820 is probably 00:18:20 and 70004 is 07:00:04. Pad it to six digits before parsing.

* **Future dates:** row 4 has a 事實發生日 of 1151113, after its announcement date of 1151106. A validation rule that rejects future dates would wrongly flag cases like this.

* **Line breaks and date styles:** the text mixes \r\n and \n, and uses both 115/11/05 and 民國115年08月24日. Normalize line breaks before the evidence check, or real quotes could be flagged as hallucinations.

* **符合條款 is not event_type:** a company name change and a factory fire are both 第51款, so one clause can cover very different events.

* **Sampling for labels:** 第31款 makes up 33% of records (132 of 401), so sample across clauses rather than purely at random.

* **Long text:** the median 說明 is 446 characters but the longest is 6,484, so decide how to handle very long ones before sending them to the LLM.
